In [ ]:
#!/usr/bin/env python
"""
LeafletFA Model Evaluation in Mouse 

This script:
1. Loads trained LeafletFA model outputs and associated data (mouse foundation smart-seq data)
2. Load the human foundation data and map junctions to human (via list of conserved junctions)
3. Apply the model to the human data
4. Save the predicted factor activities and factor usage
"""

import os
import sys
import glob
import pickle
import gzip
import warnings
from pathlib import Path
from collections import defaultdict
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from pyfaidx import Fasta

# Data analysis libraries
import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
import scipy.sparse as sp
from scipy.stats import spearmanr, pearsonr
from scipy.sparse import csr_matrix
from scipy.cluster.hierarchy import linkage, dendrogram
import scanpy as sc
import umap

# Machine learning libraries
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import (mean_squared_error, accuracy_score, r2_score, 
                           classification_report, confusion_matrix)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import resample

# Statistical modeling
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from mord import OrdinalRidge

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import ScalarFormatter
from adjustText import adjust_text

# Single-cell analysis libraries
import anndata as ad
import scanpy as sc

# Bioinformatics libraries
import gffutils
from tqdm import tqdm

# PyTorch setup
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("CUDA device name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.set_default_tensor_type("torch.FloatTensor" if device.type == "cpu" else "torch.cuda.FloatTensor")
torch.manual_seed(0)

# Configure plotting and warnings
sns.set_theme()
sc.set_figure_params(figsize=(7, 7), frameon=True, dpi=80, facecolor='white')
warnings.filterwarnings('ignore')

# =============================================================================
# Custom Module Imports
# =============================================================================

# Add custom module paths
leaflet_src_path = "/gpfs/commons/home/kisaev/LeafletFA/src/"
utils_path = "/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Multi_Species_Splicing_Foundation/shared_utils/"

for path in [leaflet_src_path, utils_path]:
    if path not in sys.path:
        sys.path.append(path)

# Import LeafletFA modules
import BetaDirichletFactor.LeafletFA as LeafletFA
import BetaDirichletFactor.utils as utilsFA

# Import utility functions
from utils import load_model
from utils import *
from figure_plotting import *
from atse_viz import *

Torch version: 2.4.1.post300
CUDA available: True
CUDA device count: 2
CUDA device name: Tesla V100-PCIE-16GB
Using device: cuda


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/torch/__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1728241823685/work/torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)


Torch Version: 2.4.1.post300
CUDA Version: 12.0
Added /gpfs/commons/home/kisaev/LeafletFA-utils to sys.path
Visualization imports successful!


In [ ]:
BASE_DIR = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION"
RESULTS_BASE_DIR = "/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/model_train/MOUSE_FOUNDATION/results"
MODEL_FILES = "/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Multi_Species_Splicing_Foundation/mouse_human_transfer_learning"

In [ ]:
# Load atse_mapping, subset_splice_adata_mouse, subset_splice_adata_human from MODEL_FILES
atse_mapping = pd.read_csv(f"{MODEL_FILES}/atse_mapping.csv")
subset_splice_adata_mouse = ad.read_h5ad(f"{MODEL_FILES}/subset_splice_adata_mouse.h5ad")
subset_splice_adata_human = ad.read_h5ad(f"{MODEL_FILES}/subset_splice_adata_human.h5ad")

In [4]:
# Main parameters to change
MODEL_TRAIN_DATE = "2025-11-19"
MODEL_ANALYSIS_DATE = "2025-11-22"
PARAM_ID = 11

# =============================================================================
# AUTO-GENERATED PATHS - DON'T EDIT BELOW THIS LINE
# =============================================================================

# Core result directories
PARAM_RESULTS_DIR = f"{RESULTS_BASE_DIR}/{MODEL_TRAIN_DATE}/{MODEL_ANALYSIS_DATE}/param_id_{PARAM_ID}"
DATA_DIR = f"{PARAM_RESULTS_DIR}/data"

# Model outputs
MODEL_OUTPUTS_DIR = f"{BASE_DIR}/Leaflet/leafletFAmodel/{MODEL_TRAIN_DATE}"
MODEL_PATH = f"{MODEL_OUTPUTS_DIR}/run_{PARAM_ID}/leafletfa_model.pkl.gz"

# Main data files from downstream analysis of model 
SPLICE_ADATA_PATH = f"{DATA_DIR}/splice_adata_PHI_psi_var_obs.h5ad"
PI_VALUES_PATH = f"{DATA_DIR}/PI_values.npy"

pi = np.load(PI_VALUES_PATH)
leaflet_model = load_model(MODEL_PATH)

Loading model to device: cuda


In [5]:
# We will run LeafletFA here but using a fixed PSI matrix 
# Check what shape and type PSI matrix need to be in 
# leaflet_model["psi_learned"].shape K by J
psi = leaflet_model["psi_learned"]
# Find the indices of the mouse junctions in the splice_adata.var["junction_id"]
psi_subset = psi[:, subset_splice_adata_mouse.var["junction_id_index"].values]
psi_input = psi_subset
K = leaflet_model["K"]
print(f"The shape of the PSI input is: {psi_input.shape}")

#Initialize model (maybe should also use fixed PI here...)
print("Initializing LeafletFA model...")
human_leaflet_model = LeafletFA.LeafletFA(
    adata=subset_splice_adata_human, 
    K=K, 
    fixed_psi=torch.tensor(psi_input), # or 
    pi_init=torch.tensor(pi),
    alpha_pi_init = torch.tensor(leaflet_model["alpha_pi"]),
    junc_specific_prior=leaflet_model["junc_specific_prior"], 
    waypoints_use=False, 
    input_conc_prior= np.inf, #leaflet_model["bb_conc"], # ideally should use the one the original model learned
    delta_fixed=torch.tensor(leaflet_model["dir_conc"]),
    num_epochs=2000, 
    print_epochs=5, 
    ELBO_num_particles=10, 
    lr=0.01, 
    gamma=0.01, 
    min_delta=10,
    num_samples=100, 
    patience=10,
    output_dir=MODEL_FILES,
    log_wandb=False  # Log to wandb
)

# Print confirm that model has dir_conc 
print(f"Model initialized with dir_conc: {human_leaflet_model.dir_conc}")
print(f"Model initialized with pi_init: {human_leaflet_model.pi_init} and alpha_pi_init: {human_leaflet_model.alpha_pi_init}")

# Train model
print(f"Extracting sparse tensors from anndata object")
human_leaflet_model.from_anndata()

print(f"Obtaining mask for sparse operations")
human_leaflet_model.initialize_triton_mask()

print("Training LeafletFA model...")
psi_init_tensor = torch.tensor(psi_input)
human_leaflet_model.train(
    num_initializations=1 #, 
    #psi_init=psi_init_tensor  
)

print("Training complete, extracting results...")
human_leaflet_model.get_all_variables()

# Update global parameters (PSI continues to evolve)
# Ensure all parameters are numpy arrays, not tensors
if hasattr(human_leaflet_model, 'psi'):
    global_psi = human_leaflet_model.psi if isinstance(human_leaflet_model.psi, np.ndarray) else human_leaflet_model.psi.cpu().numpy()
if hasattr(human_leaflet_model, 'pi'):
    global_pi = human_leaflet_model.pi if isinstance(human_leaflet_model.pi, np.ndarray) else human_leaflet_model.pi.cpu().numpy()
if hasattr(human_leaflet_model, 'alpha_pi'):
    global_alpha_pi = human_leaflet_model.alpha_pi if isinstance(human_leaflet_model.alpha_pi, np.ndarray) else human_leaflet_model.alpha_pi.cpu().numpy()
if hasattr(human_leaflet_model, 'dir_conc'):
    if isinstance(human_leaflet_model.dir_conc, torch.Tensor):
        global_dir_conc = human_leaflet_model.dir_conc.cpu().item()  # Convert to scalar
    else:
        global_dir_conc = human_leaflet_model.dir_conc
if hasattr(human_leaflet_model, 'bb_conc'):
    global_bb_conc = human_leaflet_model.bb_conc

# Store additional parameters if available
if hasattr(human_leaflet_model, 'psis_loc'):
    global_psi_loc = human_leaflet_model.psis_loc
    global_psi_scale = human_leaflet_model.psis_scale
if hasattr(human_leaflet_model, 'a'):
    global_a = human_leaflet_model.a
    global_b = human_leaflet_model.b
    global_a_shape = human_leaflet_model.a_shape
    global_a_rate = human_leaflet_model.a_rate
    global_b_shape = human_leaflet_model.b_shape
    global_b_rate = human_leaflet_model.b_rate

# Update PHI assignments for this batch of cells
if hasattr(human_leaflet_model, 'assign_post'):
    all_cell_assignments = human_leaflet_model.assign_post

# Track results
results_elbo = human_leaflet_model.best_elbo if hasattr(human_leaflet_model, 'best_elbo') else None

alpha_pi=human_leaflet_model.alpha_pi
PI = human_leaflet_model.pi
PI_df = pd.DataFrame(PI, columns=["PI"])

# 1. Get the COO matrix from the layers
coo_matrix_layer = subset_splice_adata_human.layers['Cluster_Counts']
csr_matrix_layer = coo_matrix_layer.tocsr()
subset_splice_adata_human.layers['Cluster_Counts'] = csr_matrix_layer
coo_matrix_layer = subset_splice_adata_human.layers['Junction_Counts']
csr_matrix_layer = coo_matrix_layer.tocsr()
subset_splice_adata_human.layers['Junction_Counts'] = csr_matrix_layer

# Add all_cell_assignments to subset_splice_adata_human
subset_splice_adata_human.obsm[f"X_leafletFA_K20"] = all_cell_assignments

The shape of the PSI input is: (20, 13510)
Initializing LeafletFA model...
Model initialized with dir_conc: 8.354120254516602
Model initialized with pi_init: tensor([0.0336, 0.1364, 0.1555, 0.0286, 0.0288, 0.0278, 0.0283, 0.0287, 0.0306,
        0.0880, 0.0327, 0.0289, 0.0275, 0.0346, 0.0888, 0.0264, 0.0346, 0.0451,
        0.0671, 0.0280]) and alpha_pi_init: 2.8591678142547607
Extracting sparse tensors from anndata object
Taking in the AnnData object with 76986 cells and 13510 junctions.
Processing AnnData on cuda
Data density: 4.3% (45,088,239 / 1,040,080,860)
✓ Using SPARSE computation mode (memory efficient for sparse data)
Obtaining mask for sparse operations
Training LeafletFA model...
Random seeds: [3825]
Training LeafletFA with 1 initializations.
Input concentration prior: inf
Junction-specific prior: True
Initial K to learn: 20
Random initialization of variational parameters!
-------------------------------------------------
Initialization #1 with seed 3825
-------------------

Training: 100%|██████████| 2000/2000 [53:19<00:00,  1.60s/it, loss=4.86e+08, lr=0.0001]


Training completed after 2000 epochs.
Loss plot saved to /gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/mouse_human_transfer_learning/loss_curve_seed_3825.png
Computing summary statistics for initialization #1
Initialization #1 completed with final loss: 4.8551e+08
------------------------------------------------
Model variable sizes: {'alpha_pi': torch.Size([]), 'pi': torch.Size([20]), 'assign': torch.Size([76986, 20])}
------------------------------------------------
Training complete, extracting results...
The best initialization was 0 with an ELBO of 4.8551e+08
Extracting all variables from initialization #0
Extracting PSI variational parameters...
PSI was fixed during inference. Using the provided fixed PSI instead of learned variational parameters.
PSI parameters extraction completed.
Extracting the learned beta-binomial concentration (if applicable)...
Extracting the latent representation, C x K matrix of factor activities...
Extracting the learned 

In [6]:
global_psi.shape

(20, 13510)

In [7]:
psi_init_tensor

tensor([[ 2.5931e-01, 1.2648e-251,  1.2367e-01,  ..., 9.7474e-252,
          1.5584e-01, 1.2111e-251],
        [ 2.7377e-01,  4.3650e-01,  2.1939e-01,  ...,  1.4918e-02,
          4.6610e-02,  5.3739e-01],
        [ 3.2092e-01,  1.2025e-01,  3.0538e-01,  ...,  1.1335e-02,
          7.8486e-02,  5.8707e-01],
        ...,
        [ 4.0868e-01,  8.6406e-03,  3.9006e-01,  ...,  1.3580e-01,
          9.5226e-02,  5.6011e-01],
        [ 5.2626e-01,  1.0517e-03,  5.2437e-01,  ...,  2.7693e-02,
          7.2490e-01,  9.1013e-02],
        [ 2.3090e-01,  1.1431e-01,  1.3418e-01,  ..., 9.4918e-252,
         2.9815e-252,  2.5523e-01]], dtype=torch.float64)

In [8]:
global_psi

array([[2.59313247e-001, 1.26475563e-251, 1.23671197e-001, ...,
        9.74736584e-252, 1.55836265e-001, 1.21107438e-251],
       [2.73766631e-001, 4.36497006e-001, 2.19393212e-001, ...,
        1.49182163e-002, 4.66104157e-002, 5.37390073e-001],
       [3.20918124e-001, 1.20248443e-001, 3.05382165e-001, ...,
        1.13354287e-002, 7.84861237e-002, 5.87067559e-001],
       ...,
       [4.08684681e-001, 8.64057482e-003, 3.90059001e-001, ...,
        1.35799980e-001, 9.52264662e-002, 5.60105773e-001],
       [5.26258976e-001, 1.05166838e-003, 5.24367915e-001, ...,
        2.76932240e-002, 7.24898573e-001, 9.10130261e-002],
       [2.30904769e-001, 1.14313068e-001, 1.34176368e-001, ...,
        9.49183092e-252, 2.98148314e-252, 2.55225300e-001]])

In [9]:
# add psi_learned to .varm
subset_splice_adata_human.varm["psi_learned"] = global_psi.T

In [10]:
subset_splice_adata_human.write_h5ad(f"{MODEL_FILES}/subset_splice_adata_human_with_mouse_transfer.h5ad", compression="gzip")

In [11]:
pi

array([0.03362913, 0.13641767, 0.1555055 , 0.02857082, 0.02878112,
       0.02779431, 0.02827007, 0.02872326, 0.03055756, 0.08795834,
       0.03273541, 0.02892375, 0.02747601, 0.03459625, 0.08883404,
       0.02639075, 0.03459676, 0.04512753, 0.06713581, 0.02797593],
      dtype=float32)

In [12]:
PI_df

,PI
0,0.075569
1,0.045696
2,0.050982
3,0.039284
4,0.041228
5,0.041846
6,0.037120
7,0.056091
8,0.058976
9,0.049401


In [13]:
# save new PI_df
PI_df.to_csv(f"{MODEL_FILES}/PI_df_human_with_mouse_transfer.csv", index=False)